# CIC-IDS setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

CIC_IDS_PATH = "../data/raw/CIC-IDS-2017"

## Load the dataset

In [ ]:
# 1. Find and load all CSV files in the directory
all_files = glob.glob(os.path.join(CIC_IDS_DIR, "*.csv"))
df_list = []

print(f"Found {len(all_files)} CSV files. Loading and concatenating...")
for file in all_files:
    df = pd.read_csv(file)
    df_list.append(df)

# 2. Merge them into a single dataset
df_cic = pd.concat(df_list, ignore_index=True)

# 3. Clean headers (remove leading/trailing spaces)
df_cic.columns = df_cic.columns.str.strip()

# 4. Convert to Binary Labels (0 = BENIGN, 1 = Attack)
df_cic['Binary_Label'] = df_cic['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

# 5. Separate features and labels
X_cic = df_cic.drop(columns=['Label', 'Binary_Label'])
y_cic = df_cic['Binary_Label'].values

print(f"Total samples across all days: {df_cic.shape[0]}")
print(f"Features per sample: {X_cic.shape[1]}")
print(f"Benign: {(y_cic == 0).sum()}")
print(f"Malware/Attack: {(y_cic == 1).sum()}")

## Label distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = ["Benign", "Malware/Attack"]

counts = [
    (y_cic == 0).sum(), 
    (y_cic == 1).sum()
]

ax.bar(labels, counts, color=["steelblue", "tomato"])
ax.set_title("CIC-IDS dataset - label distribution")
ax.set_ylabel("Samples")

# Add text labels above bars
for i, v in enumerate(counts):
    # Dynamic offset based on max count
    ax.text(i, v + (max(counts) * 0.02), str(v), ha="center", fontweight="bold") 

plt.tight_layout()
plt.savefig("../../../experiments/cic_label_distribution.png", dpi=150)
plt.show()

## Features statistics

In [ ]:
# Taking a subset of samples to view statistics (similar to your EMBER notebook)
n_samples = 2000

print(f"Statistics on the first {n_samples} samples:\n")
print(X_cic.head(n_samples).describe().round(3))

## Features distribution per class

In [ ]:
# compare between the first 5 features for benign and malware
n_features = 5
fig, axes = plt.subplots(1, n_features, figsize=(4 * n_features, 6))

if n_features == 1:
    axes = [axes]  # wrap it so the loop still works
axes = axes.flatten()

# Extract values and limit to first 2000 instances to speed up plotting
X_ben = X_cic[y_cic == 0].values[:2000]
X_mal = X_cic[y_cic == 1].values[:2000]

for i, ax in enumerate(axes):
    # CIC-IDS sometimes contains Infinity or NaN values (e.g., Flow Bytes/s). 
    # np.nan_to_num safely converts these to finite numbers for plotting
    ben_vals = np.nan_to_num(X_ben[:, i])
    mal_vals = np.nan_to_num(X_mal[:, i])
    
    ax.hist(ben_vals, bins=30, alpha=0.6,
            color="steelblue", label="Benign", density=True)
    ax.hist(mal_vals, bins=30, alpha=0.6,
            color="tomato", label="Malware", density=True)
    
    # Grab the actual column name for the title
    feat_name = X_cic.columns[i]
    ax.set_title(feat_name[:20] + '..' if len(feat_name) > 20 else feat_name)
    ax.legend(fontsize=7)

plt.suptitle(f"Distribution of the first {n_features} features per class (CIC-IDS)", y=1.02)
plt.tight_layout()
plt.savefig("../../../experiments/cic_feature_distributions.png", dpi=150)
plt.show()